In [62]:
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Sampler
import wandb
from sklearn.metrics import precision_score
from accelerate import Accelerator
from accelerate import DistributedType
import os
from utils.utils import seed_everything
from transformers import LongformerTokenizer
from datasets import EHR_Longformer_Dataset
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm
import torch.nn.functional as F
from models.longformernormal import LongformerPretrainNormal
from torch.optim.lr_scheduler import LinearLR, SequentialLR, ExponentialLR, LambdaLR, CosineAnnealingWarmRestarts
# from pretrain_train import train
import logging
import sys
from torch.utils.data.distributed import DistributedSampler
from torch.optim.lr_scheduler import _LRScheduler
import math
from torch.nn.utils.rnn import pad_sequence
import random
import math
from typing import Any, Optional
from transformers import AutoModel, AutoTokenizer
import numpy as np
import torch
from torch import nn
from transformers import LongformerConfig, BigBirdConfig
import warnings

In [1]:
import pandas as pd
val = pd.read_pickle('datasets/send_finetune_dataset/finetune_val_24.pkl')
test = pd.read_pickle('datasets/send_finetune_dataset/finetune_test_24.pkl')

In [2]:
print(len(val.keys()), len(test.keys()))

2098 2100


In [63]:

class CosineAnnealingWarmupRestarts(_LRScheduler):
    """
        optimizer (Optimizer): Wrapped optimizer.
        first_cycle_steps (int): First cycle step size.
        cycle_mult(float): Cycle steps magnification. Default: -1.
        max_lr(float): First cycle's max learning rate. Default: 0.1.
        min_lr(float): Min learning rate. Default: 0.001.
        warmup_steps(int): Linear warmup step size. Default: 0.
        gamma(float): Decrease rate of max learning rate by cycle. Default: 1.
        last_epoch (int): The index of last epoch. Default: -1.
    """
    
    def __init__(self,
                 optimizer : torch.optim.Optimizer,
                 first_cycle_steps : int,
                 cycle_mult : float = 1.,
                 max_lr : float = 0.1,
                 min_lr : float = 0.001,
                 warmup_steps : int = 0,
                 gamma : float = 1.,
                 last_epoch : int = -1
        ):
        assert warmup_steps < first_cycle_steps
        
        self.first_cycle_steps = first_cycle_steps # first cycle step size
        self.cycle_mult = cycle_mult # cycle steps magnification
        self.base_max_lr = max_lr # first max learning rate
        self.max_lr = max_lr # max learning rate in the current cycle
        self.min_lr = min_lr # min learning rate
        self.warmup_steps = warmup_steps # warmup step size
        self.gamma = gamma # decrease rate of max learning rate by cycle
        
        self.cur_cycle_steps = first_cycle_steps # first cycle step size
        self.cycle = 0 # cycle count
        self.step_in_cycle = last_epoch # step size of the current cycle
        
        super(CosineAnnealingWarmupRestarts, self).__init__(optimizer, last_epoch)
        
        # set learning rate min_lr
        self.init_lr()
    
    def init_lr(self):
        self.base_lrs = []
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.min_lr
            self.base_lrs.append(self.min_lr)
    
    def get_lr(self):
        if self.step_in_cycle == -1:
            return self.base_lrs
        elif self.step_in_cycle < self.warmup_steps:
            return [(self.max_lr - base_lr)*self.step_in_cycle / self.warmup_steps + base_lr for base_lr in self.base_lrs]
        else:
            return [base_lr + (self.max_lr - base_lr) \
                    * (1 + math.cos(math.pi * (self.step_in_cycle-self.warmup_steps) \
                                    / (self.cur_cycle_steps - self.warmup_steps))) / 2
                    for base_lr in self.base_lrs]

    def step(self, epoch=None):
        if epoch is None:
            epoch = self.last_epoch + 1
            self.step_in_cycle = self.step_in_cycle + 1
            if self.step_in_cycle >= self.cur_cycle_steps:
                self.cycle += 1
                self.step_in_cycle = self.step_in_cycle - self.cur_cycle_steps
                self.cur_cycle_steps = int((self.cur_cycle_steps - self.warmup_steps) * self.cycle_mult) + self.warmup_steps
        else:
            if epoch >= self.first_cycle_steps:
                if self.cycle_mult == 1.:
                    self.step_in_cycle = epoch % self.first_cycle_steps
                    self.cycle = epoch // self.first_cycle_steps
                else:
                    n = int(math.log((epoch / self.first_cycle_steps * (self.cycle_mult - 1) + 1), self.cycle_mult))
                    self.cycle = n
                    self.step_in_cycle = epoch - int(self.first_cycle_steps * (self.cycle_mult ** n - 1) / (self.cycle_mult - 1))
                    self.cur_cycle_steps = self.first_cycle_steps * self.cycle_mult ** (n)
            else:
                self.cur_cycle_steps = self.first_cycle_steps
                self.step_in_cycle = epoch
                
        self.max_lr = self.base_max_lr * (self.gamma**self.cycle)
        self.last_epoch = math.floor(epoch)
        for param_group, lr in zip(self.optimizer.param_groups, self.get_lr()):
            param_group['lr'] = lr


def configure_optimizers(model, args):
    optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate)
    
    for param_group in optimizer.param_groups:
        param_group['initial_lr'] = args.learning_rate
    
    # n_warmup_steps = int(n_steps * 0.1)
    # n_decay_steps = n_steps - n_warmup_steps
    
    # warmup = LinearLR(optimizer, 
    #                     start_factor=0.01,
    #                     end_factor=1.0,
    #                     total_iters=n_warmup_steps)
    
    # decay = LinearLR(optimizer,
    #                     start_factor=1.0,
    #                     end_factor=0.01,
    #                     total_iters=n_decay_steps)
    
    # scheduler = SequentialLR(optimizer, 
    #                             schedulers=[warmup, decay],
    #                             milestones=[n_warmup_steps])
    # scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=500, T_mult=2)
    if args.resume:
        scheduler = CosineAnnealingWarmupRestarts(optimizer,
                                                first_cycle_steps=2690,
                                                cycle_mult=1.5,
                                                max_lr=0.0001,
                                                min_lr=0.000001,
                                                warmup_steps=269,
                                                gamma=0.9,
                                                last_epoch=args.resume_epoch
                                                )
    else:
        scheduler = CosineAnnealingWarmupRestarts(optimizer,
                                                first_cycle_steps=2690,
                                                cycle_mult=1.5,
                                                max_lr=0.0001,
                                                min_lr=0.000001,
                                                warmup_steps=269,
                                                gamma=0.9,
                                                )

    return optimizer, {"scheduler": scheduler, "interval": "step"}

In [64]:
parser = argparse.ArgumentParser()

# Required parameters
parser.add_argument("--exp_name", type=str, default="pretrain")
parser.add_argument("--save_path", type=str, default="./results")
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--checkpoint_dir", type=str, default="./checkpoints")

# Model parameters
parser.add_argument("--vocab_size", type=int, default=50265)
parser.add_argument("--itemid_size", type=int, default=3893)
parser.add_argument("--unit_size", type=int, default=81)
parser.add_argument("--gender_size", type=int, default=2)
parser.add_argument("--continuous_size", type=int, default=3)
parser.add_argument("--task_size", type=int, default=20)
parser.add_argument("--name_size", type=int, default=35)
parser.add_argument("--description_size", type=int, default=12)
parser.add_argument("--max_position_embeddings", type=int, default=5000)
parser.add_argument("--max_age", type=int, default=100)
parser.add_argument("--batch_size", type=int, default=2)
parser.add_argument("--pin_memory", type=bool, default=True)
parser.add_argument("--nodes", type=int, default=1)
parser.add_argument("--gpus", type=int, default=2)
parser.add_argument("--start_epoch", type=int, default=0)
parser.add_argument("--epochs", type=int, default=300)
parser.add_argument("--log_every_n_steps", type=int, default=100)
parser.add_argument("--acc", type=int, default=8)
parser.add_argument("--resume_checkpoint", type=str, default=None)
parser.add_argument("--num_workers", type=int, default=4)      
parser.add_argument("--embedding_size", type=int, default=768)
parser.add_argument("--num_hidden_layers", type=int, default=1)
parser.add_argument("--num_attention_heads", type=int, default=1)
parser.add_argument("--intermediate_size", type=int, default=3072)
parser.add_argument("--learning_rate", type=float, default=1e-4)
parser.add_argument("--dropout_prob", type=float, default=0.1)
parser.add_argument("--layer_norm_eps", type=float, default=1e-6)
parser.add_argument("--device", type=str, default="cuda")
parser.add_argument("--gpu_mixed_precision", type=bool, default=True)
parser.add_argument("--patience", type=int, default=50)
parser.add_argument("--clip_interval", type=int, default=200)
parser.add_argument("--resume", type=bool, default=False)
parser.add_argument("--resume_epoch", type=int, default=0)
parser.add_argument("--loss_alpha", type=float, default=0.5)
parser.add_argument("--regression_mode", type=bool, default=False)
parser.add_argument("--similarity_mode", type=bool, default=False)
parser.add_argument("--similarity_factor", type=float, default=0.25)



args = parser.parse_args([])
args.attention_window = [512] * args.num_hidden_layers


In [65]:
seed_everything(args.seed)

tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")

itemid2idx = pd.read_pickle("datasets/new_label2idx.pkl")
unit2idx = pd.read_pickle("datasets/new_unit2idx.pkl")
idx2label = pd.read_pickle("datasets/new_idx2label.pkl") ##############


Seed set to 42


In [66]:
len(itemid2idx)

3893

In [67]:
df = pd.read_pickle("datasets/pretrain_train.pkl")

In [68]:
train_dataset = EHR_Longformer_Dataset(Path("./datasets"), "train", tokenizer, itemid2idx, unit2idx, vocab_size=args.itemid_size, use_itemid=True, mode='pretrain', mask_mode='mlm', mask_ratio=0.15)
valid_dataset = EHR_Longformer_Dataset(Path("./datasets"), "valid", tokenizer, itemid2idx, unit2idx, vocab_size=args.itemid_size, use_itemid=True, mode='pretrain', mask_mode='mlm', mask_ratio=0.15)

In [69]:
# import traceback

# try:
#     print("Dataset 초기화 시작")
#     train_dataset = EHR_Longformer_Dataset(Path("./datasets"), "train", tokenizer, itemid2idx, unit2idx, vocab_size=args.itemid_size, use_itemid=True, mode='pretrain', mask_mode='mlm', mask_ratio=0.15)
#     print("train_dataset 생성 완료")
    
#     valid_dataset = EHR_Longformer_Dataset(Path("./datasets"), "valid", tokenizer, itemid2idx, unit2idx, vocab_size=args.itemid_size, use_itemid=True, mode='pretrain', mask_mode='mlm', mask_ratio=0.15)
#     print("valid_dataset 생성 완료")

# except Exception as e:                                      
#     print("예외 발생:", e)
#     print("예외 타입:", type(e).__name__)
#     print("스택 트레이스:")
#     import traceback
#     traceback.print_exc()

In [70]:
len(train_dataset)

42519

In [71]:
train_dataset[0][-4:][0][:55]

tensor([3, 5, 3, 4, 3, 6, 6, 6, 6, 3, 3, 3, 4, 3, 3, 4, 4, 3, 3, 3, 3, 3, 3, 3,
        3, 3, 3, 3, 3, 3, 3, 3, 3, 9, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 4,
        3, 4, 3, 3, 4, 9, 6])

In [72]:
train_dataset[0][-3:][0][:55]

tensor([4, 5, 3, 4, 3, 6, 6, 6, 6, 3, 4, 3, 3, 3, 3, 7, 7, 3, 3, 3, 3, 3, 3, 3,
        3, 3, 3, 3, 3, 4, 3, 3, 4, 8, 3, 4, 4, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 3,
        3, 3, 3, 3, 3, 8, 6])

In [78]:
train_loader = DataLoader(train_dataset, 
                            batch_size=args.batch_size,
                            shuffle=True,  # shuffle should be False if using DistributedSampler
                            # sampler=train_sampler,
                            pin_memory=args.pin_memory, 
                            num_workers=args.num_workers,
                            drop_last=True,
                            )


In [79]:
additional_tokens = torch.tensor([1, 1, 1]).unsqueeze(0).repeat(args.batch_size, 1).to('cuda')

for step, batch in tqdm(enumerate(train_loader)):
    batch = tuple(t.to(args.device) for t in batch)
    input_ids, attention_mask, age_ids, gender_ids, value_ids, unit_ids, offset_ids, position_ids, token_type_ids, ordercategoryname_ids, ordercategorydescription_ids, task_ids, labels = batch
    if additional_tokens.shape[0] != labels.shape[0]:
        print("additional_tokens", additional_tokens.shape)
        print("labels", labels.shape)
        print("additional_tokens", additional_tokens)
        print("labels", labels)
        print(step)
    labels = torch.cat([additional_tokens, labels], dim=1)
    

print(input_ids.shape)
print(attention_mask.shape)
print(age_ids.shape)
print(gender_ids.shape)
print(value_ids.shape)
print(unit_ids.shape)
print(offset_ids.shape)
print(position_ids.shape)
print(token_type_ids.shape)
print(ordercategoryname_ids.shape)
print(ordercategorydescription_ids.shape)
print(task_ids.shape)
print(labels.shape)

print(input_ids.to('cuda'))

21259it [02:10, 162.49it/s]

torch.Size([2, 4093])
torch.Size([2, 4093])
torch.Size([2, 1])
torch.Size([2, 1])
torch.Size([2, 4093])
torch.Size([2, 4093])
torch.Size([2, 4093])
torch.Size([2, 4093])
torch.Size([2, 4093])
torch.Size([2, 4093])
torch.Size([2, 4093])
torch.Size([2, 1])
torch.Size([2, 4096])
tensor([[290,   7,   4,  ...,   0,   0,   0],
        [  7,   9,   4,  ...,   0,   0,   0]], device='cuda:0')


: 

In [18]:
input_ids.min(), input_ids.max()

(tensor(0, device='cuda:0'), tensor(3470, device='cuda:0'))

In [19]:
unit_ids.min(), unit_ids.max()

(tensor(0, device='cuda:0'), tensor(77, device='cuda:0'))

In [20]:
value_ids.min(), value_ids.max()

(tensor(-500., device='cuda:0'), tensor(2500., device='cuda:0'))

In [21]:
offset_ids.min(), offset_ids.max()

(tensor(-500., device='cuda:0'), tensor(1.9875, device='cuda:0'))

In [22]:
position_ids.min(), position_ids.max()

(tensor(0, device='cuda:0'), tensor(115, device='cuda:0'))

In [23]:
ordercategorydescription_ids.min(), ordercategorydescription_ids.max()

(tensor(0, device='cuda:0'), tensor(10, device='cuda:0'))

In [24]:
ordercategoryname_ids.min(), ordercategoryname_ids.max()

(tensor(0, device='cuda:0'), tensor(26, device='cuda:0'))

In [25]:
input_ids

tensor([[405, 404, 841,  ...,   0,   0,   0],
        [890,  38,  35,  ...,   0,   0,   0]], device='cuda:0')

In [26]:
from transformers import LongformerConfig

config = LongformerConfig(
            vocab_size = args.vocab_size,
            max_position_embeddings=args.max_position_embeddings,
            hidden_size=args.embedding_size,
            num_hidden_layers=args.num_hidden_layers,
            num_attention_heads=args.num_attention_heads,
            intermediate_size=args.intermediate_size,
            hidden_dropout_prob=args.dropout_prob, 
            attention_probs_dropout_prob=args.dropout_prob,
            attention_window=[512] * args.num_hidden_layers,
            # # layer_norm_eps=self.layer_norm_eps
        )

In [27]:
# import traceback
# from models.embedding import EHREmbedding
# from models.embedding import ConceptEmbeddingwithClinicalBert, PositionalEmbedding, TimeEmbedding, ValueEmbedding, UnitEmbedding, AgeEmbedding, GenderEmbedding, TaskEmbedding, OrderCategoryNameEmbedding, OrderCategoryDescriptionEmbedding
# import math
# try:
#     # clinicalbert = ClinicalBERTEmbedding()
#     # concept_embedding = ConceptEmbeddingwithClinicalBert(idx2label, clinicalbert).to(args.device)
#     concept_embedding = ConceptEmbeddingwithClinicalBert(idx2label).to(args.device)
#     positional_embedding = PositionalEmbedding(config.max_position_embeddings, config.hidden_size).to(args.device)
#     time_embedding = TimeEmbedding(1, config.hidden_size).to(args.device)
#     value_embedding = ValueEmbedding(config.hidden_size).to(args.device)
#     unit_embedding = UnitEmbedding(args.unit_size, config.hidden_size).to(args.device)
#     age_embedding = AgeEmbedding(args.max_age, config.hidden_size).to(args.device)
#     gender_embedding = GenderEmbedding(args.gender_size, config.hidden_size).to(args.device)
#     task_embedding = TaskEmbedding(args.task_size, config.hidden_size).to(args.device)
#     ordercategoryname_embedding = OrderCategoryNameEmbedding(args.name_size, config.hidden_size).to(args.device)
#     ordercategorydescription_embedding = OrderCategoryDescriptionEmbedding(args.description_size, config.hidden_size).to(args.device)

# except Exception as e:                                      
#     print("예외 발생:", e)
#     print("예외 타입:", type(e).__name__)
#     print("스택 트레이스:")
#     import traceback
#     traceback.print_exc()

In [28]:
# input_ids = input_ids.to('cuda')
# value_ids = value_ids.to('cuda')
# unit_ids = unit_ids.to('cuda')
# time_ids = offset_ids.to('cuda')
# position_ids = position_ids.to('cuda')
# token_type_ids = token_type_ids.to('cuda')
# ordername_ids = ordercategoryname_ids.to('cuda')
# orderdescription_ids = ordercategorydescription_ids.to('cuda')
# age_ids = age_ids.to('cuda')
# gender_ids = gender_ids.to('cuda')
# task_ids = task_ids.to('cuda')

In [29]:
input_ids.shape

torch.Size([2, 4093])

In [30]:
input_ids

tensor([[405, 404, 841,  ...,   0,   0,   0],
        [890,  38,  35,  ...,   0,   0,   0]], device='cuda:0')

In [31]:
ordercategorydescription_ids

tensor([[3, 3, 3,  ..., 0, 0, 0],
        [3, 3, 3,  ..., 0, 0, 0]], device='cuda:0')

In [32]:
max(ordercategorydescription_ids[0])


tensor(10, device='cuda:0')

In [33]:
max(ordercategorydescription_ids[1])

tensor(10, device='cuda:0')

In [34]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [35]:
# max_ordername = 0
# for step, batch in tqdm(enumerate(train_loader)):
#     batch = tuple(t.to(args.device) for t in batch)
#     input_ids, attention_mask, age_ids, gender_ids, value_ids, unit_ids, offset_ids, position_ids, token_type_ids, ordercategoryname_ids, ordercategorydescription_ids, task_ids, labels = batch
#     labels = torch.cat([additional_tokens, labels], dim=1)
#     try: 
#         [[idx2orderdescription[idx.item()] for idx in seq] for seq in ordercategorydescription_ids]
#     except Exception as e:
#         print(e)

 


In [36]:
input_ids

tensor([[405, 404, 841,  ...,   0,   0,   0],
        [890,  38,  35,  ...,   0,   0,   0]], device='cuda:0')

In [37]:
ordercategorydescription_ids

tensor([[3, 3, 3,  ..., 0, 0, 0],
        [3, 3, 3,  ..., 0, 0, 0]], device='cuda:0')

In [38]:
[[idx2label[ids.item()] for ids in seq] for seq in input_ids.cpu()]

[['Apnea Interval',
  'Fspn High',
  'Inspiratory Time',
  'Mean Airway Pressure',
  'Minute Volume',
  'Minute Volume Alarm - High',
  'Minute Volume Alarm - Low',
  'PEEP set',
  'Paw High',
  'Peak Insp. Pressure',
  'Respiratory Rate (Set)',
  'MASK',
  'Respiratory Rate (spontaneous)',
  'Tidal Volume (observed)',
  'Tidal Volume (set)',
  'Vti High',
  'Respiratory Rate',
  'Heart Rate',
  'MASK',
  'Heart rate Alarm - High',
  'Non-Invasive Blood Pressure Alarm - High',
  'Non-Invasive Blood Pressure Alarm - Low',
  'Tandem Heart Inflow Line',
  'MASK',
  'MASK',
  'MASK',
  'MASK',
  '18 Gauge',
  '16 Gauge',
  'O2 saturation pulseoxymetry',
  'Respiratory Rate',
  'Non Invasive Blood Pressure diastolic',
  'Non Invasive Blood Pressure mean',
  'Non Invasive Blood Pressure systolic',
  'Heart Rate',
  'O2 saturation pulseoxymetry',
  'Respiratory Rate',
  'Temperature Fahrenheit',
  'Pre-Admission/Non-ICU Intake',
  'NaCl 0.9%',
  'Propofol',
  'Solution',
  'Arterial Base Exce

In [39]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [40]:
ordercategorydescription_ids

tensor([[3, 3, 3,  ..., 0, 0, 0],
        [3, 3, 3,  ..., 0, 0, 0]], device='cuda:0')

In [41]:
import traceback
from models.embedding import EHREmbedding
from models.embedding import ConceptEmbeddingwithClinicalBert, PositionalEmbedding, TimeEmbedding, ValueEmbedding, UnitEmbedding, AgeEmbedding, GenderEmbedding, TaskEmbedding, OrderCategoryNameEmbedding, OrderCategoryDescriptionEmbedding
import math
try:
    # clinicalbert = ClinicalBERTEmbedding()
    # concept_embedding = ConceptEmbeddingwithClinicalBert(idx2label, clinicalbert).to(args.device)
    concept_embedding = ConceptEmbeddingwithClinicalBert(idx2label).to(args.device)
    positional_embedding = PositionalEmbedding(config.max_position_embeddings, config.hidden_size).to(args.device)
    time_embedding = TimeEmbedding(1, config.hidden_size).to(args.device)
    value_embedding = ValueEmbedding(config.hidden_size).to(args.device)
    unit_embedding = UnitEmbedding(args.unit_size, config.hidden_size).to(args.device)
    age_embedding = AgeEmbedding(args.max_age, config.hidden_size).to(args.device)
    gender_embedding = GenderEmbedding(args.gender_size, config.hidden_size).to(args.device)
    task_embedding = TaskEmbedding(args.task_size, config.hidden_size).to(args.device)
    ordercategoryname_embedding = OrderCategoryNameEmbedding(args.name_size, config.hidden_size).to(args.device)
    ordercategorydescription_embedding = OrderCategoryDescriptionEmbedding(args.description_size, config.hidden_size).to(args.device)

except Exception as e:                                      
    print("예외 발생:", e)
    print("예외 타입:", type(e).__name__)
    print("스택 트레이스:")
    import traceback
    traceback.print_exc()

/home/DAHS3/anaconda3/envs/sj/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [50]:
try:
    input_embeds = concept_embedding(input_ids)
    value_embeds = value_embedding(value_ids)
    gender_embeds = gender_embedding(gender_ids)
    age_embeds = age_embedding(age_ids)
    position_embeds = positional_embedding(position_ids)
    time_embeds = time_embedding(offset_ids)
    unit_embeds = unit_embedding(unit_ids)
    task_embeds = task_embedding(task_ids)
    ordercategoryname_embeds = ordercategoryname_embedding(ordercategoryname_ids)
    ordercategorydescription_embeds = ordercategorydescription_embedding(ordercategorydescription_ids)


except Exception as e:                                      
    print("예외 발생:", e)
    print("예외 타입:", type(e).__name__)
    print("스택 트레이스:")
    import traceback
    traceback.print_exc()



# unit_embeds = unit_embedding(unit_ids)
# time_embeds = time_embedding(time_ids)

In [52]:
print(input_embeds.shape)
print(value_embeds.shape)
print(gender_embeds.shape)
print(age_embeds.shape)
print(position_embeds.shape)
print(time_embeds.shape)
print(unit_embeds.shape)
print(task_embeds.shape)
print(ordercategoryname_embeds.shape)
print(ordercategorydescription_embeds.shape)

torch.Size([2, 4093, 768])
torch.Size([2, 4093, 768])
torch.Size([2, 1, 768])
torch.Size([2, 1, 768])
torch.Size([2, 4093, 768])
torch.Size([2, 4093, 768])
torch.Size([2, 4093, 768])
torch.Size([2, 1, 768])
torch.Size([2, 4093, 768])
torch.Size([2, 4093, 768])


In [53]:
value_embeds

tensor([[[-7.8514e-01,  1.8028e+00,  6.4218e+00,  ..., -4.8068e-01,
          -9.3974e+00, -2.0463e+00],
         [-4.2287e-01,  9.2807e-01,  3.0917e+00,  ..., -1.2264e-01,
          -4.7327e+00, -9.0687e-01],
         [-1.7499e-01,  1.0384e-01, -1.4316e-01,  ...,  1.1851e-01,
          -1.4092e-01,  4.6840e-04],
         ...,
         [-1.1572e-01,  1.2859e-01, -2.8445e-01,  ...,  6.2114e-02,
           1.2749e-01, -2.6308e-01],
         [-1.1572e-01,  1.2859e-01, -2.8445e-01,  ...,  6.2114e-02,
           1.2749e-01, -2.6308e-01],
         [-1.1572e-01,  1.2859e-01, -2.8445e-01,  ...,  6.2114e-02,
           1.2749e-01, -2.6308e-01]],

        [[-1.6974e-01,  2.2566e-01,  3.4310e-01,  ...,  9.1162e-02,
          -9.1457e-01, -3.9970e-02],
         [-1.1661e+00,  2.7297e+00,  9.9743e+00,  ..., -8.5385e-01,
          -1.4369e+01, -3.2551e+00],
         [-7.5065e+00,  1.8128e+01,  6.9027e+01,  ..., -7.0478e+00,
          -9.7032e+01, -2.3354e+01],
         ...,
         [-1.1572e-01,  1

In [54]:
position_embeds

tensor([[[8.4147e-01, 5.6009e-01, 8.1525e-01,  ..., 1.0000e+00,
          1.0491e-08, 1.0000e+00],
         [8.4147e-01, 5.6009e-01, 8.1525e-01,  ..., 1.0000e+00,
          1.0491e-08, 1.0000e+00],
         [8.4147e-01, 5.6009e-01, 8.1525e-01,  ..., 1.0000e+00,
          1.0491e-08, 1.0000e+00],
         ...,
         [0.0000e+00, 1.0000e+00, 0.0000e+00,  ..., 1.0000e+00,
          0.0000e+00, 1.0000e+00],
         [0.0000e+00, 1.0000e+00, 0.0000e+00,  ..., 1.0000e+00,
          0.0000e+00, 1.0000e+00],
         [0.0000e+00, 1.0000e+00, 0.0000e+00,  ..., 1.0000e+00,
          0.0000e+00, 1.0000e+00]],

        [[8.4147e-01, 5.6009e-01, 8.1525e-01,  ..., 1.0000e+00,
          1.0491e-08, 1.0000e+00],
         [8.4147e-01, 5.6009e-01, 8.1525e-01,  ..., 1.0000e+00,
          1.0491e-08, 1.0000e+00],
         [8.4147e-01, 5.6009e-01, 8.1525e-01,  ..., 1.0000e+00,
          1.0491e-08, 1.0000e+00],
         ...,
         [0.0000e+00, 1.0000e+00, 0.0000e+00,  ..., 1.0000e+00,
          0.000

In [55]:
time_embeds

tensor([[[ 0.7426, -0.3283, -0.1765,  ...,  0.3314, -0.4361,  0.7617],
         [ 0.7426, -0.3283, -0.1765,  ...,  0.3314, -0.4361,  0.7617],
         [ 0.7426, -0.3283, -0.1765,  ...,  0.3314, -0.4361,  0.7617],
         ...,
         [ 0.7427, -0.3277, -0.1768,  ...,  0.3315, -0.4362,  0.7613],
         [ 0.7427, -0.3277, -0.1768,  ...,  0.3315, -0.4362,  0.7613],
         [ 0.7427, -0.3277, -0.1768,  ...,  0.3315, -0.4362,  0.7613]],

        [[ 0.7424, -0.3289, -0.1761,  ...,  0.3313, -0.4360,  0.7621],
         [ 0.7424, -0.3289, -0.1761,  ...,  0.3313, -0.4360,  0.7621],
         [ 0.7424, -0.3289, -0.1761,  ...,  0.3313, -0.4360,  0.7621],
         ...,
         [ 0.7427, -0.3277, -0.1768,  ...,  0.3315, -0.4362,  0.7613],
         [ 0.7427, -0.3277, -0.1768,  ...,  0.3315, -0.4362,  0.7613],
         [ 0.7427, -0.3277, -0.1768,  ...,  0.3315, -0.4362,  0.7613]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [56]:
unit_embeds

tensor([[[-0.4151,  0.1219, -0.0775,  ..., -0.4694,  1.1547, -0.0336],
         [-0.3124,  0.2020,  0.7281,  ..., -0.8240, -0.5125, -1.1466],
         [-0.4151,  0.1219, -0.0775,  ..., -0.4694,  1.1547, -0.0336],
         ...,
         [ 1.4411,  1.0507, -0.2597,  ..., -0.6980, -0.9960,  1.1429],
         [ 1.4411,  1.0507, -0.2597,  ..., -0.6980, -0.9960,  1.1429],
         [ 1.4411,  1.0507, -0.2597,  ..., -0.6980, -0.9960,  1.1429]],

        [[-0.8145,  0.2314, -0.6050,  ...,  0.2019, -2.5446, -0.3550],
         [ 0.3076, -1.2160,  0.3590,  ...,  0.0340,  1.4458,  0.7409],
         [ 0.3076, -1.2160,  0.3590,  ...,  0.0340,  1.4458,  0.7409],
         ...,
         [ 1.4411,  1.0507, -0.2597,  ..., -0.6980, -0.9960,  1.1429],
         [ 1.4411,  1.0507, -0.2597,  ..., -0.6980, -0.9960,  1.1429],
         [ 1.4411,  1.0507, -0.2597,  ..., -0.6980, -0.9960,  1.1429]]],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)

In [57]:
# print(input_embeds.shape)
print(value_embeds.shape)
# print(time_embeds.shape)
# print(unit_embeds.shape)
# print(position_embeds.shape)
# print(ordercategoryname_embeds.shape)
# print(ordercategorydescription_embeds.shape)

torch.Size([2, 4093, 768])


In [58]:
embeddings = input_embeds + time_embeds +  position_embeds + value_embeds + unit_embeds + ordercategoryname_embeds + ordercategorydescription_embeds
embeddings.shape

torch.Size([2, 4093, 768])

In [59]:
import traceback
from models.embedding import EHREmbedding
from models.embedding import ConceptEmbeddingwithClinicalBert, PositionalEmbedding, TimeEmbedding, ValueEmbedding, UnitEmbedding, AgeEmbedding, GenderEmbedding, TaskEmbedding, OrderCategoryNameEmbedding, OrderCategoryDescriptionEmbedding
import math
from typing import Any, Optional
from transformers import AutoModel, AutoTokenizer
import numpy as np
import torch
from torch import nn
from transformers import LongformerConfig, BigBirdConfig
try:
    embeddings = EHREmbedding(
            config=config,
            itemid_size=args.itemid_size,
            unit_size=args.unit_size,
            max_age=100,
            max_len=args.max_position_embeddings,
            gender_size=args.gender_size,
            task_size=args.task_size,
            idx2label=idx2label, ###########
            # idx2ordername=idx2ordername,
            # idx2orderdescription=idx2orderdescription,      
            name_size=args.name_size,
            description_size=args.description_size,
            use_itemid=True,
            inputs_embeds=None,
        ).to(args.device)


except Exception as e:                                      
    print("예외 발생:", e)
    print("예외 타입:", type(e).__name__)
    print("스택 트레이스:")
    import traceback
    traceback.print_exc()

In [60]:
import traceback
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_SHOW_CPP_STACKTRACES'] = "1"
try:
    combined_embed = embeddings(
            input_ids = input_ids,
            value_ids = value_ids,
            unit_ids = unit_ids,
            time_ids = offset_ids,
            position_ids = position_ids,
            token_type_ids = token_type_ids,
            ordername_ids = ordercategoryname_ids,
            orderdescription_ids = ordercategorydescription_ids,
            age_ids = age_ids,
            gender_ids = gender_ids,
            task_ids = task_ids
        )


except Exception as e:                                      
    print("예외 발생:", e)
    print("예외 타입:", type(e).__name__)
    print("스택 트레이스:")
    import traceback
    traceback.print_exc()

In [61]:
from transformers import LongformerConfig

config = LongformerConfig(
            vocab_size = args.vocab_size,
            max_position_embeddings=args.max_position_embeddings,
            hidden_size=args.embedding_size,
            num_hidden_layers=args.num_hidden_layers,
            num_attention_heads=args.num_attention_heads,
            intermediate_size=args.intermediate_size,
            hidden_dropout_prob=args.dropout_prob, 
            attention_probs_dropout_prob=args.dropout_prob,
            attention_window=[512] * args.num_hidden_layers,
            # # layer_norm_eps=self.layer_norm_eps
        )
class TimeEmbedding(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(TimeEmbedding, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        self.w = nn.Parameter(torch.randn(input_dim, output_dim))
        self.b = nn.Parameter(torch.randn(output_dim))
        self.freqs = nn.Parameter(torch.randn(output_dim))
        
        self.projection = nn.Linear(2*output_dim, output_dim)
        
    def forward(self, x):
        x = x.unsqueeze(-1)
        w = self.w.unsqueeze(0)
        b = self.b.unsqueeze(0).unsqueeze(0)
        feqs = self.freqs.unsqueeze(0).unsqueeze(0)
        
        linear = torch.matmul(x, w) + b
        linear = linear.squeeze(-2)
        
        periodic = torch.sin(torch.matmul(x, feqs) + b)
        periodic = periodic.squeeze(-2)
        
        combined = torch.cat([linear, periodic], dim=-1)
        
        return self.projection(combined)
    
        
class ValueEmbedding(nn.Module):
    """ Embedding layer for value features """
    
    def __init__(self, hidden_size):
        super().__init__()
        self.value_embedding = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size)
        )
        
    def forward(self, x):
        x = x.unsqueeze(-1)
        return self.value_embedding(x)
    
class UnitEmbedding(nn.Module):
    def __init__(self, unit_size, embedding_size):
        super().__init__()
        self.unit_embedding = nn.Embedding(unit_size, embedding_size)
    
    def forward(self, x):
        return self.unit_embedding(x)
    
class PositionalEmbedding(nn.Module):
    """ Embedding layer for position features """
    
    def __init__(self, max_len:int, embedding_size:int):
        super().__init__()
        
        self.position_embeddings = nn.Embedding(
            max_len, embedding_size, padding_idx=0
        ).from_pretrained(self._init_posi_embedding(max_len, embedding_size))
    
    def _init_posi_embedding(self, max_len, hidden_size):
        def even_code(pos, idx):
            return np.sin(pos / (10000 ** (2 * idx / hidden_size)))
        
        def odd_code(pos, idx):
            return np.cos(pos / (10000 ** (2 * idx / hidden_size)))
        
        # initialize position embedding table
        lookup_table = np.zeros((max_len, hidden_size), dtype=np.float32)
        
        # reset table parameters with hard encoding
        # set even dimension
        for pos in range(max_len):
            for idx in np.arange(0, hidden_size, step=2):
                lookup_table[pos, idx] = even_code(pos, idx)
            
        # set odd dimension
        for pos in range(max_len):
            for idx in np.arange(1, hidden_size, step=2):
                lookup_table[pos, idx] = odd_code(pos, idx)
                
        return torch.tensor(lookup_table)
        
    def forward(self, x):
        """ Apply positional embedding"""
        return self.position_embeddings(x)
    
# class ContinuousEmbedding(nn.Module):
#     def __init__(self, continuous_size, embedding_size):
#         super().__init__()
#         self.embedding_size = embedding_size
#         self.embedding = nn.Embedding(continuous_size, embedding_size)
        
#     def forward(self, x):
#         return self.embedding(x)
    

class AgeEmbedding(nn.Module):
    def __init__(self, max_age, embedding_size):
        super().__init__()
        self.embedding_size = embedding_size
        self.age_embedding = nn.Embedding(max_age, embedding_size)
    
    def forward(self, x):
        return self.age_embedding(x)
    
class GenderEmbedding(nn.Module):
    def __init__(self, gender_size, embedding_size):
        super().__init__()
        self.embedding_size = embedding_size
        self.gender_embedding = nn.Embedding(gender_size, embedding_size)
    
    def forward(self, x):
        return self.gender_embedding(x)
    
class TaskEmbedding(nn.Module):
    def __init__(self, task_size, embedding_size):
        super().__init__()
        self.embedding_size = embedding_size
        self.task_embedding = nn.Embedding(task_size, embedding_size)
    
    def forward(self, x):
        return self.task_embedding(x)
    
# class ConceptEmbedding(nn.Module):
#     def __init__(self, vocab_size, itemid_size, embedding_size, padding_idx=1, use_itemid=True):
#         super().__init__()
#         self.embedding_size = embedding_size
#         self.use_itemid = use_itemid
#         self.padding_idx = padding_idx

#         if use_itemid:  
#             self.procedure_embedding = nn.Embedding(itemid_size, embedding_size, padding_idx=self.padding_idx)
#             self.medication_embedding = nn.Embedding(itemid_size, embedding_size, padding_idx=self.padding_idx)
#             self.chart_embedding = nn.Embedding(itemid_size, embedding_size, padding_idx=self.padding_idx)
#         else:
#             self.procedure_embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=self.padding_idx)
#             self.medication_embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=self.padding_idx)
#             self.chart_embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=self.padding_idx)
        
        
    
#     def forward(self, concept, token_type):
        
#         if not self.use_itemid:
#             batch_size, seq_length = concept.size()[:2]
#             label_embed = torch.zeros(batch_size, seq_length, self.procedure_embedding.embedding_dim, dtype=torch.float32, device=concept.device)
        
#         else:
#             batch_size, seq_length = concept.size()
#             label_embed = torch.zeros(batch_size, seq_length, self.procedure_embedding.embedding_dim, dtype=torch.float32, device=concept.device)
        
#         procedure_mask = token_type == 1
#         medication_mask = token_type == 2
#         chart_mask = token_type == 3
        
#         if not self.use_itemid:
#             if procedure_mask.any():
#                 procedure_ids = concept[procedure_mask]
#                 procedure_embeds = self.procedure_embedding(procedure_ids)
#                 label_embed[procedure_mask] = procedure_embeds.mean(dim=1)

#             if medication_mask.any():
#                 medication_ids = concept[medication_mask]
#                 medication_embeds = self.medication_embedding(medication_ids)
#                 label_embed[medication_mask] = medication_embeds.mean(dim=1)
            
#             if chart_mask.any():
#                 chart_ids = concept[chart_mask]
#                 chart_embeds = self.chart_embedding(chart_ids)
#                 label_embed[chart_mask] = chart_embeds.mean(dim=1)  
                
#         else:
#             if procedure_mask.any():
#                 procedure_indices = procedure_mask.nonzero(as_tuple=False)
#                 procedure_ids = concept[procedure_mask]
#                 procedure_embeds = self.procedure_embedding(procedure_ids)
                
#                 for idx, (batch_idx, seq_idx) in enumerate(procedure_indices):
#                     label_embed[batch_idx, seq_idx] = procedure_embeds[idx]

#             if medication_mask.any():
#                 medication_indices = medication_mask.nonzero(as_tuple=False)
#                 medication_ids = concept[medication_mask]
#                 medication_embeds = self.medication_embedding(medication_ids)
                
#                 for idx, (batch_idx, seq_idx) in enumerate(medication_indices):
#                     label_embed[batch_idx, seq_idx] = medication_embeds[idx]
            
#             if chart_mask.any():
#                 chart_indices = chart_mask.nonzero(as_tuple=False)
#                 chart_ids = concept[chart_mask]
#                 chart_embeds = self.chart_embedding(chart_ids)
                
#                 for idx, (batch_idx, seq_idx) in enumerate(chart_indices):
#                     label_embed[batch_idx, seq_idx] = chart_embeds[idx]
                
#         return label_embed
    
class ConceptEmbeddingwithClinicalBert(nn.Module):
    def __init__(self, idx2label, model_name="emilyalsentzer/Bio_ClinicalBERT"):
        super().__init__()
        self.idx2label = idx2label
        self.embedding_model = AutoModel.from_pretrained(model_name).to("cuda")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        for param in self.embedding_model.parameters():
            param.requires_grad = False
        
    def get_embedding_batch(self, text):
        tokens = self.tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(self.embedding_model.device)
        
        outputs = self.embedding_model(**tokens)
        embeddings = outputs.last_hidden_state[:, 0, :]
        # embeddings = outputs.hidden_state[:, 1:-1, :].mean(dim=1) 
        # embeddings = embeddings.cpu().clone().detach()

        return embeddings
    
    # def get_embedding_batch(self, text, sub_batch_size=8):
    #     embeddings_list = []
    #     for i in range(0, len(text), sub_batch_size):
    #         sub_text = text[i:i + sub_batch_size]
    #         tokens = self.tokenizer(
    #             sub_text,
    #             padding=True,
    #             truncation=True,
    #             max_length=128,
    #             return_tensors="pt"
    #         ).to(self.embedding_model.device)

            
    #         outputs = self.embedding_model(**tokens)
    #         embeddings = outputs.last_hidden_state[:, 0, :]  # CLS 토큰
    #         embeddings_list.append(embeddings)

    #     return torch.cat(embeddings_list, dim=0)
        
    def forward(self, concept):
        input2label = [[self.idx2label[ids.item()] for ids in seq] for seq in concept.cpu()]
        label_embeddings = []
        for labels in input2label:
            with torch.no_grad():
                embeddings =self.get_embedding_batch(labels)
            label_embeddings.append(embeddings)
        return torch.stack(label_embeddings).to(self.embedding_model.device)
    
class OrderCategoryNameEmbedding(nn.Module):
    def __init__(self, idx2label, model_name="emilyalsentzer/Bio_ClinicalBERT"):
        super().__init__()
        self.idx2label = idx2label
        self.embedding_model = AutoModel.from_pretrained(model_name).to("cuda")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        for param in self.embedding_model.parameters():
            param.requires_grad = False
            
    def get_embedding_batch(self, text):
        tokens = self.tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(self.embedding_model.device)
        
        outputs = self.embedding_model(**tokens)
        embeddings = outputs.last_hidden_state[:, 0, :]
        # embeddings = outputs.hidden_state[:, 1:-1, :].mean(dim=1) 
        # embeddings = embeddings.cpu().clone().detach()

        return embeddings
    
    def forward(self, concept):
        input2label = [[self.idx2label[ids.item()] for ids in seq] for seq in concept.cpu()]
        label_embeddings = []
        for labels in input2label:
            with torch.no_grad():
                embeddings =self.get_embedding_batch(labels)
            label_embeddings.append(embeddings)
        return torch.stack(label_embeddings).to(self.embedding_model.device)
    
class OrderCategoryDescriptionEmbedding(nn.Module):
    def __init__(self, idx2label, model_name="emilyalsentzer/Bio_ClinicalBERT"):
        super().__init__()
        self.idx2label = idx2label
        self.embedding_model = AutoModel.from_pretrained(model_name).to("cuda")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        for param in self.embedding_model.parameters():
            param.requires_grad = False
            
    def get_embedding_batch(self, text):
        tokens = self.tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(self.embedding_model.device)
        
        outputs = self.embedding_model(**tokens)
        embeddings = outputs.last_hidden_state[:, 0, :]
        # embeddings = outputs.hidden_state[:, 1:-1, :].mean(dim=1) 
        # embeddings = embeddings.cpu().clone().detach()

        return embeddings
    
    def forward(self, concept):
        input2label = [[self.idx2label[ids.item()] for ids in seq] for seq in concept.cpu()]
        label_embeddings = []
        for labels in input2label:
            with torch.no_grad():
                embeddings =self.get_embedding_batch(labels)
            label_embeddings.append(embeddings)
        return torch.stack(label_embeddings).to(self.embedding_model.device)
            
        
    

# class EHREmbedding(nn.Module):
#     def __init__(self, 
#                  config: LongformerConfig, 
#                  itemid_size, 
#                  unit_size,
#                  max_age, 
#                  max_len, 
#                  continuous_size, 
#                  gender_size, 
#                  task_size, 
#                  idx2label, #########
#                  padding_idx=0, 
#                  use_itemid=True, 
#                  inputs_embeds=None):
#         super().__init__()
#         self.vocab_size = config.vocab_size
#         self.itemid_size = itemid_size
#         self.unit_size = unit_size
#         self.max_age = max_age
#         self.hidden_size = config.hidden_size
#         self.continuous_size = continuous_size
#         self.gender_size = gender_size
#         self.task_size = task_size
#         self.padding_idx = padding_idx
#         self.use_itemid = use_itemid
#         self.max_position_embeddings = max_len
#         self.inputs_embeds = inputs_embeds 
#         self.idx2label = idx2label #####
        
#         # self.concept_embedding = ConceptEmbedding(self.vocab_size, self.itemid_size, self.hidden_size, padding_idx, use_itemid) ####
#         self.concept_embedding = ConceptEmbeddingwithClinicalBert(self.idx2label)
#         self.position_embedding = PositionalEmbedding(self.max_position_embeddings, self.hidden_size)
#         self.time_embedding = TimeEmbedding(1, self.hidden_size)
#         self.value_embedding = ValueEmbedding(self.hidden_size)
#         self.unit_embedding = UnitEmbedding(self.unit_size, self.hidden_size)
#         self.continuous_embedding = ContinuousEmbedding(self.continuous_size, self.hidden_size)
#         self.age_embedding = AgeEmbedding(self.max_age, self.hidden_size)
#         self.gender_embedding = GenderEmbedding(self.gender_size, self.hidden_size)
#         self.task_embedding = TaskEmbedding(self.task_size, self.hidden_size)
        
#         self.LayerNorm = torch.nn.LayerNorm(self.hidden_size, eps=1e-6)
#         self.dropout = nn.Dropout(config.hidden_dropout_prob)
        
#     # def cache_input(
#     #     self,
#     #     value_ids: torch.Tensor,
#     #     unit_ids: torch.Tensor,
#     #     time_ids: torch.Tensor,
#     #     continuous_ids: torch.Tensor,
#     #     age_ids: torch.Tensor,
#     #     gender_ids: torch.Tensor,
#     #     task_token: torch.Tensor,
#     # ):
#     #     self.value_ids = value_ids
#     #     self.unit_ids = unit_ids
#     #     self.time_ids = time_ids
#     #     self.continuous_ids = continuous_ids
#     #     self.age_ids = age_ids
#     #     self.gender_ids = gender_ids
#     #     self.task_ids = task_token
    
#     # def clear_cache(self):
#     #     del self.value_ids, self.unit_ids, self.time_ids, self.continuous_ids, self.age_ids, self.gender_ids, self.task_ids
    
    
#     def forward(self, 
#                 input_ids: torch.Tensor,
#                 value_ids: torch.Tensor,
#                 unit_ids: torch.Tensor,
#                 time_ids: torch.Tensor,
#                 continuous_ids: torch.Tensor,
#                 position_ids: torch.Tensor,
#                 token_type_ids: torch.Tensor,
#                 age_ids: torch.Tensor,
#                 gender_ids: torch.Tensor,
#                 task_ids: torch.Tensor,
#                 inputs_embeds=None,
#                 ):
        
#         if inputs_embeds is not None:
#             return inputs_embeds

#         # concept_embed = self.concept_embedding(input_ids, token_type_ids)
#         concept_embed = self.concept_embedding(input_ids)
#         time_embed = self.time_embedding(time_ids)
#         age_embed = self.age_embedding(age_ids)
#         gender_embed = self.gender_embedding(gender_ids)
#         positional_embed = self.position_embedding(position_ids)
   
        
#         value_embed = torch.zeros(concept_embed.size(), dtype=torch.float32, device=input_ids.device)
#         unit_embed = torch.zeros(concept_embed.size(), dtype=torch.float32, device=input_ids.device)
#         continuous_embed = torch.zeros(concept_embed.size(), dtype=torch.float32, device=input_ids.device)
#         # continuous_embed = self.continuous_embedding(continuous_ids)
        
#         value_mask = (token_type_ids == 2) | (token_type_ids == 3)
#         continuous_mask = (token_type_ids == 1) | (token_type_ids == 2)
        
    
        
#         # if value_mask.any():
#         #     value_indices = value_mask.nonzero(as_tuple=False)
#         #     value_ids = value_ids[value_mask]
#         #     value_embeds = self.value_embedding(value_ids)
#         #     unit_ids = unit_ids[value_mask]
#         #     unit_embeds = self.unit_embedding(unit_ids)
            
#         #     for idx, (batch_idx, seq_idx) in enumerate(value_indices):
#         #         value_embed[batch_idx, seq_idx] = value_embeds[idx]
#         #         unit_embed[batch_idx, seq_idx] = unit_embeds[idx]
        
#         # if continuous_mask.any():
#         #     continuous_indices = continuous_mask.nonzero(as_tuple=False)
#         #     continuous_ids = continuous_ids[continuous_mask]
#         #     continuous_embeds = self.continuous_embedding(continuous_ids)
            
#         #     for idx, (batch_idx, seq_idx) in enumerate(continuous_indices):
#         #         continuous_embed[batch_idx, seq_idx] = continuous_embeds[idx]

#         if value_mask.any():
#             # value_indices = value_mask.nonzero(as_tuple=False)
#             # value_ids = value_ids[value_mask]
#             # value_embeds = self.value_embedding(value_ids)
#             # unit_ids = unit_ids[value_mask]
#             # unit_embeds = self.unit_embedding(unit_ids)
            
#             # for idx, (batch_idx, seq_idx) in enumerate(value_indices):
#             #     value_embed[batch_idx, seq_idx] = value_embeds[idx]
#             #     unit_embed[batch_idx, seq_idx] = unit_embeds[idx]
#             value_embeds = self.value_embedding(value_ids[value_mask])
#             unit_embeds = self.unit_embedding(unit_ids[value_mask])
            
#             # Use scatter for efficient assignment
#             # value_embed = value_embed.masked_scatter_(value_mask.unsqueeze(-1), value_embeds)
#             value_embed = value_embed.masked_scatter_(value_mask.unsqueeze(-1), value_embeds.to(value_embed.dtype))
#             # unit_embed = unit_embed.masked_scatter_(value_mask.unsqueeze(-1), unit_embeds)
#             unit_embed = unit_embed.masked_scatter_(value_mask.unsqueeze(-1), unit_embeds.to(unit_embed.dtype))
                
#         if continuous_mask.any():
#             # continuous_indices = continuous_mask.nonzero(as_tuple=False)
#             # continuous_ids = continuous_ids[continuous_mask]
#             # continuous_embeds = self.continuous_embedding(continuous_ids)
            
#             # for idx, (batch_idx, seq_idx) in enumerate(continuous_indices):
#             #     continuous_embed[batch_idx, seq_idx] = continuous_embeds[idx]
#             continuous_embeds = self.continuous_embedding(continuous_ids[continuous_mask])
    
#             # Use scatter for efficient assignment
#             # continuous_embed = continuous_embed.masked_scatter_(continuous_mask.unsqueeze(-1), continuous_embeds)
#             continuous_embed = continuous_embed.masked_scatter_(continuous_mask.unsqueeze(-1), continuous_embeds.to(continuous_embed.dtype))
        
        
#         task_embed = self.task_embedding(task_ids)
        
#         # if torch.isinf(concept_embed).any():
#         #     print("concept embed inf ok")
#         # if torch.isinf(time_embed).any():
#         #     print("time embed inf ok")
#         # if torch.isinf(positional_embed).any():
#         #     print("position embed inf ok")
#         if torch.isinf(value_embed).any():
#             print("value_embed inf ok")
#         # if torch.isinf(unit_embed).any():
#         #     print("unit embed inf ok")
#         # if torch.isinf(continuous_embed).any():
#         #     print("continuous embed inf ok")
        
#         embeddings = concept_embed + time_embed + positional_embed + value_embed + unit_embed + continuous_embed
        
#         combined_embed = torch.cat((task_embed, age_embed, gender_embed, embeddings), dim=1)
#         # if torch.isinf(combined_embed).any():
#         #     print("combined embed inf ok")
#         #     print(torch.max(combined_embed), torch.min(combined_embed))
#         combined_embed = self.LayerNorm(combined_embed)
#         # if torch.isnan(combined_embed).any():
#         #     print("embedding layernorm nan ok")
#         #     print(torch.max(combined_embed), torch.min(combined_embed))
#         combined_embed = self.dropout(combined_embed)
#         # if torch.isnan(combined_embed).any():
#         #     print("embedding dropout nan ok")
        
#         # self.clear_cache()
        
#         # print(combined_embed.shape)
        
#         return combined_embed
    

        
                

class EHREmbedding(nn.Module):
    def __init__(self, 
                 config: LongformerConfig, 
                 itemid_size, 
                 unit_size,
                 max_age, 
                 max_len, 
                 gender_size, 
                 task_size, 
                 idx2label, #########
                 idx2ordername,
                 idx2orderdescription,
                 padding_idx=0, 
                 use_itemid=True, 
                 inputs_embeds=None):
        super().__init__()
        self.vocab_size = config.vocab_size
        self.itemid_size = itemid_size
        self.unit_size = unit_size
        self.max_age = max_age
        self.hidden_size = config.hidden_size
        self.gender_size = gender_size
        self.task_size = task_size
        self.padding_idx = padding_idx
        self.use_itemid = use_itemid
        self.max_position_embeddings = max_len
        self.inputs_embeds = inputs_embeds 
        self.idx2label = idx2label #####
        self.idx2ordername = idx2ordername
        self.idx2orderdescription = idx2orderdescription
        
        # self.concept_embedding = ConceptEmbedding(self.vocab_size, self.itemid_size, self.hidden_size, padding_idx, use_itemid) ####
        self.concept_embedding = ConceptEmbeddingwithClinicalBert(self.idx2label)
        self.position_embedding = PositionalEmbedding(self.max_position_embeddings, self.hidden_size).to("cuda")
        self.time_embedding = TimeEmbedding(1, self.hidden_size).to("cuda")
        self.value_embedding = ValueEmbedding(self.hidden_size).to("cuda")
        self.unit_embedding = UnitEmbedding(self.unit_size, self.hidden_size).to("cuda")
        # self.continuous_embedding = ContinuousEmbedding(self.continuous_size, self.hidden_size)
        self.age_embedding = AgeEmbedding(self.max_age, self.hidden_size).to("cuda")
        self.gender_embedding = GenderEmbedding(self.gender_size, self.hidden_size).to("cuda")
        self.task_embedding = TaskEmbedding(self.task_size, self.hidden_size).to("cuda")
        self.ordername_embedding = OrderCategoryNameEmbedding(self.idx2ordername)
        self.orderdescription_embedding = OrderCategoryDescriptionEmbedding(self.idx2orderdescription)
        
        self.LayerNorm = torch.nn.LayerNorm(self.hidden_size, eps=1e-6)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        
    # def cache_input(
    #     self,
    #     value_ids: torch.Tensor,
    #     unit_ids: torch.Tensor,
    #     time_ids: torch.Tensor,
    #     continuous_ids: torch.Tensor,
    #     age_ids: torch.Tensor,
    #     gender_ids: torch.Tensor,
    #     task_token: torch.Tensor,
    # ):
    #     self.value_ids = value_ids
    #     self.unit_ids = unit_ids
    #     self.time_ids = time_ids
    #     self.continuous_ids = continuous_ids
    #     self.age_ids = age_ids
    #     self.gender_ids = gender_ids
    #     self.task_ids = task_token
    
    # def clear_cache(self):
    #     del self.value_ids, self.unit_ids, self.time_ids, self.continuous_ids, self.age_ids, self.gender_ids, self.task_ids
    
    
    def forward(self, 
                input_ids: torch.Tensor,
                value_ids: torch.Tensor,
                unit_ids: torch.Tensor,
                time_ids: torch.Tensor,
                # continuous_ids: torch.Tensor,
                position_ids: torch.Tensor,
                token_type_ids: torch.Tensor,
                ordername_ids: torch.Tensor,
                orderdescription_ids: torch.Tensor,
                age_ids: torch.Tensor,
                gender_ids: torch.Tensor,
                task_ids: torch.Tensor,
                inputs_embeds=None,
                ):
        
        if inputs_embeds is not None:
            return inputs_embeds

        # concept_embed = self.concept_embedding(input_ids, token_type_ids)
        concept_embed = self.concept_embedding(input_ids)
        time_embed = self.time_embedding(time_ids)
        age_embed = self.age_embedding(age_ids)
        gender_embed = self.gender_embedding(gender_ids)
        positional_embed = self.position_embedding(position_ids)
   
        
        # value_embed = torch.zeros(concept_embed.size(), dtype=torch.float32, device=input_ids.device)
        # unit_embed = torch.zeros(concept_embed.size(), dtype=torch.float32, device=input_ids.device)
        value_embed = self.value_embedding(value_ids)
        unit_embed = self.unit_embedding(unit_ids)

        ordername_embed = self.ordername_embedding(ordername_ids)
        orderdescription_embed = self.orderdescription_embedding(orderdescription_ids)
                
        task_embed = self.task_embedding(task_ids)
        
        # if torch.isinf(concept_embed).any():
        #     print("concept embed inf ok")
        # if torch.isinf(time_embed).any():
        #     print("time embed inf ok")
        # if torch.isinf(positional_embed).any():
        #     print("position embed inf ok")
        if torch.isinf(value_embed).any():
            print("value_embed inf ok")
        # if torch.isinf(unit_embed).any():
        #     print("unit embed inf ok")
        # if torch.isinf(continuous_embed).any():
        #     print("continuous embed inf ok")
        
        embeddings = concept_embed + time_embed + positional_embed + value_embed + unit_embed + ordername_embed + orderdescription_embed
        
        combined_embed = torch.cat((task_embed, age_embed, gender_embed, embeddings), dim=1)
        # if torch.isinf(combined_embed).any():
        #     print("combined embed inf ok")
        #     print(torch.max(combined_embed), torch.min(combined_embed))
        combined_embed = self.LayerNorm(combined_embed)
        # if torch.isnan(combined_embed).any():
        #     print("embedding layernorm nan ok")
        #     print(torch.max(combined_embed), torch.min(combined_embed))
        combined_embed = self.dropout(combined_embed)
        # if torch.isnan(combined_embed).any():
        #     print("embedding dropout nan ok")
        
        # self.clear_cache()
        
        # print(combined_embed.shape)
        
        return combined_embed
    

        
                
                

In [13]:
idx2unit = pd.read_pickle("datasets/new_idx2valueuom.pkl")

In [14]:
len(idx2unit)

75

In [15]:
position_ids

tensor([[1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')

In [16]:



concept_embedding = ConceptEmbeddingwithClinicalBert(idx2label)
position_embedding = PositionalEmbedding(512, 768).to('cuda')
time_embedding = TimeEmbedding(1, 768).to('cuda')
value_embedding = ValueEmbedding(768).to('cuda')
unit_embedding = UnitEmbedding(75, 768).to('cuda')
age_embedding = AgeEmbedding(100, 768).to('cuda')
gender_embedding = GenderEmbedding(2, 768).to('cuda')
task_embedding = TaskEmbedding(2, 768).to('cuda')
ordername_embedding = OrderCategoryNameEmbedding(idx2ordername)
orderdescription_embedding = OrderCategoryDescriptionEmbedding(idx2orderdescription)


concept_embed = concept_embedding(input_ids)
position_embed = position_embedding(position_ids)
time_embed = time_embedding(offset_ids)
value_embed = value_embedding(value_ids)
unit_embed = unit_embedding(unit_ids)
age_embed = age_embedding(age_ids)
gender_embed = gender_embedding(gender_ids)
task_embed = task_embedding(task_ids)
ordername_embed = ordername_embedding(ordercategoryname_ids)
orderdescription_embed = orderdescription_embedding(ordercategorydescription_ids)

c:\Users\sj\anaconda3\envs\torch_py39\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [14]:

ordername_embedding = OrderCategoryNameEmbedding(idx2ordername)
orderdescription_embedding = OrderCategoryDescriptionEmbedding(idx2orderdescription)

ordername_embed = ordername_embedding(ordercategoryname_ids)
orderdescription_embed = orderdescription_embedding(ordercategorydescription_ids)

c:\Users\sj\anaconda3\envs\torch_py39\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


AttributeError: 'PositionalEmbedding' object has no attribute 'device'

In [29]:
concept_embed

tensor([[[ 0.0224, -0.5075,  0.0349,  ..., -0.2002, -0.6451, -0.1660],
         [ 0.1883,  0.2157,  0.0840,  ..., -0.2434, -0.0336, -0.2626],
         [ 0.0523,  0.1777,  0.1889,  ..., -0.1901,  0.5592, -0.1311],
         ...,
         [ 0.3749,  0.3069, -0.1242,  ..., -0.2289,  0.2238,  0.0739],
         [ 0.3749,  0.3069, -0.1242,  ..., -0.2289,  0.2238,  0.0739],
         [ 0.3749,  0.3069, -0.1242,  ..., -0.2289,  0.2238,  0.0739]],

        [[ 0.0523,  0.1777,  0.1889,  ..., -0.1901,  0.5592, -0.1311],
         [ 0.1169,  0.0608,  0.1303,  ...,  0.1981, -0.6585, -0.4879],
         [ 0.4986,  0.3942, -0.3069,  ..., -0.1482,  0.3038,  0.0628],
         ...,
         [ 0.7346,  0.4911,  0.1346,  ..., -0.0415,  0.7357, -0.2757],
         [ 0.3410,  0.0604,  0.3821,  ...,  0.0818,  0.1421, -0.1738],
         [ 0.4455,  0.1414,  0.1027,  ...,  0.2510,  0.0406, -0.1065]]],
       device='cuda:0')

In [30]:
concept_embedding = ConceptEmbeddingwithClinicalBert(idx2label)

concept_embed = concept_embedding(input_ids)

In [27]:
position_em

torch.Size([2, 4093, 768])